# BTS Digital Twin (NVS) — Vòng 1: train baseline 3DGS (1 scene / lần chạy)

Notebook này chạy **Vòng 1 (baseline)** của kiến trúc train nhiều vòng — xem
`docs/00_MASTER_PLAN.md` mục 3.1. Chỉ xử lý **1 scene duy nhất mỗi lần chạy** (biến
`SCENE` ở Bước 5). Có **7 scene** (xem `pipeline/common/scenes.py`): 5 scene BTS —
`HCM0421`, `HCM0539`, `HCM0540`, `HCM0644`, `HCM0674` — và 2 scene tổng quát —
`bonsai`, `chair`.

Cách dùng: đổi `SCENE` ở Bước 5 rồi Save Version, lặp lại cho cả 7 scene (có thể chạy
song song nhiều version). Mỗi scene độc lập hoàn toàn.

**KHÔNG scene nào có ảnh ground-truth thật cho `test/`** (BTC chỉ cấp
`test_poses.csv`) — nhưng notebook này VẪN ra được điểm số nội bộ (PSNR/SSIM/LPIPS/
Score) nhờ tự tách **holdout** từ chính ảnh train (`00_make_holdout_split.py`) — xem
biến `MODE` ở Bước 5. `MODE="holdout"` dùng để đo Score (KHÔNG dùng để nộp bài),
`MODE="final"` dùng để train checkpoint nộp bài thật (100% ảnh train, không ra Score).

Cấu hình Vòng 1 GIỮ ĐƠN GIẢN theo đúng kết luận đo thật ở repo tiền nhiệm (xem
`docs/PORTED_KNOWLEDGE.md` mục 2): chỉ có `ANTIALIASING` (mip-splatting, mặc định
BẬT — đã đo có lợi). KHÔNG có depth-prior/antenna-focus/exposure-comp ở Vòng 1 (đã đo
thật KHÔNG cải thiện Score tổng) — các kỹ thuật đó dành cho notebook Vòng 2+
(`kaggle_round2_refine.ipynb`, do agent khác phụ trách).

**Trước khi chạy, cần điền:**
1. Settings → Accelerator: **GPU T4 x2** (hoặc P100) → Internet: **On**.
2. `REPO_URL`/`GIT_BRANCH` ở Bước 3, `GDRIVE_URL` ở Bước 4 (đã điền sẵn), các biến ở
   Bước 5 (`SCENE`, `MODE`, `ANTIALIASING`).
3. Ước lượng thời gian train (30000 iterations mặc định) tuỳ độ nặng scene — theo
   dõi log `pipeline/work/<scene>/02_train_baseline.log`, canh đủ trong 1 session Kaggle.

**Bảo mật:** để notebook này **Private**.

## Bước 1 — Cài đặt

In [ ]:
import torch, subprocess, sys
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG có GPU — vào Settings bật Accelerator GPU trước khi chạy tiếp")
print("Torch:", torch.__version__, "| CUDA build:", torch.version.cuda)

In [ ]:
!pip install -q pycolmap "scikit-image>=0.19" lpips plyfile tqdm gdown

## Bước 2 — Clone + build 3D Gaussian Splatting

Repo gốc `graphdeco-inria/gaussian-splatting` — dùng để train/render, không tự viết lại
trainer (quá nhiều chi tiết dễ sai: densification, SH coefficients...). Pin đúng commit
`54c035f7834b564019656c3e3fcc3646292f727d` (đã xác nhận có `--antialiasing`, xem
`docs/00_MASTER_PLAN.md` mục 3.1). Bước build 2 CUDA extension
(`diff-gaussian-rasterization`, `simple-knn`) mất khoảng 2-5 phút.

In [ ]:
%cd /kaggle/working
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting.git
%cd /kaggle/working/gaussian-splatting
!git checkout 54c035f7834b564019656c3e3fcc3646292f727d
!git submodule update --init --recursive
%cd /kaggle/working
!pip install -q ./gaussian-splatting/submodules/diff-gaussian-rasterization
!pip install -q ./gaussian-splatting/submodules/simple-knn

import os
os.environ["GS_REPO"] = "/kaggle/working/gaussian-splatting"
print("GS_REPO =", os.environ["GS_REPO"])

## Bước 3 — Lấy code pipeline từ Git repo của bạn (khuyến nghị để **Private**)

Repo Private vẫn clone được bình thường trên Kaggle, chỉ cần xác thực bằng
**Personal Access Token (PAT)** thay vì mật khẩu. Các bước 1 lần:

1. Đẩy code lên GitHub, chọn **Private** khi tạo repo (không ai ngoài bạn xem được,
   kể cả khi bạn share notebook Kaggle này cho người khác sau này).
2. Tạo token: GitHub → **Settings → Developer settings → Personal access tokens →
   Fine-grained tokens → Generate new token**. Chọn:
   - Repository access: **Only select repositories** → chọn đúng repo vừa tạo.
   - Permissions → Contents: **Read-only** (không cần quyền gì khác).
   - Đặt ngày hết hạn (Expiration) ngắn thôi, vd 30-90 ngày — hết hạn thì tạo token mới.
3. **Copy token, dán vào Kaggle Secrets (KHÔNG dán thẳng vào code)**: trong notebook
   Kaggle, vào menu **Add-ons → Secrets → Add a new secret** → Label đặt đúng tên
   `GITHUB_TOKEN`, Value dán token vừa copy → Save. Cell bên dưới sẽ tự đọc secret
   này lúc chạy, token không hề xuất hiện trong code/notebook — kể cả nếu lỡ share
   notebook cho người khác, họ cũng không nhìn thấy được token của bạn.

Cell dò-thư-mục bên dưới tự tìm thư mục con tên `pipeline` (chứa `common/` và
`scripts/`) ở bất kỳ độ sâu nào trong repo vừa clone, không cần đúng ngay gốc repo.

In [ ]:
REPO_URL = "https://github.com/ThongLuc2k3/BTS-Digital-Twin-MultiRound.git"
GIT_BRANCH = "main"  # <-- đổi nếu code Vòng 1 đang nằm ở nhánh khác chưa merge vào main

# GITHUB_TOKEN: ưu tiên lấy từ Kaggle Secrets (an toàn, không lộ trong code).
# Chỉ cần dán thẳng vào biến bên dưới nếu bạn KHÔNG dùng Kaggle Secrets (kém an
# toàn hơn — token sẽ nằm lộ trong notebook, đừng share notebook cho ai nếu làm vậy).
GITHUB_TOKEN = ""

try:
    from kaggle_secrets import UserSecretsClient
    if not GITHUB_TOKEN:
        GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
        print("Đã lấy GITHUB_TOKEN từ Kaggle Secrets.")
except Exception:
    if not GITHUB_TOKEN:
        print("Không tìm thấy Kaggle Secret 'GITHUB_TOKEN' (bỏ qua nếu repo Public, "
              "hoặc bạn đã dán token thẳng vào biến GITHUB_TOKEN ở trên).")

assert REPO_URL, "Chưa điền REPO_URL — dán link git repo chứa thư mục pipeline/ vào biến này rồi chạy lại cell."

clone_url = REPO_URL
if GITHUB_TOKEN and "github.com" in REPO_URL:
    clone_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")

!rm -rf /kaggle/working/_repo_clone
!git clone --depth 1 -b "{GIT_BRANCH}" "{clone_url}" /kaggle/working/_repo_clone

In [ ]:
# Tự dò thư mục "pipeline" (chứa common/ và scripts/) ở bất kỳ đâu trong repo vừa
# clone, rồi symlink về /kaggle/working/pipeline — mọi cell sau đều gọi script từ đây.
import os
import shutil
from pathlib import Path

CLONE_ROOT = Path("/kaggle/working/_repo_clone")
found = None
for dirpath, dirnames, filenames in os.walk(CLONE_ROOT):
    p = Path(dirpath)
    if p.name == "pipeline" and "common" in dirnames and "scripts" in dirnames:
        found = p
        break
if found is None and (CLONE_ROOT / "common").exists() and (CLONE_ROOT / "scripts").exists():
    found = CLONE_ROOT  # trường hợp bạn push thẳng NỘI DUNG pipeline/ làm gốc repo

if found is None:
    raise SystemExit(
        "Không tìm thấy thư mục 'pipeline' (chứa common/ và scripts/) trong repo vừa clone.\n"
        f"Nội dung clone nằm ở {CLONE_ROOT} — kiểm tra lại đã push đúng thư mục pipeline/ lên git chưa."
    )

print("Tìm thấy code pipeline tại:", found)
target = Path("/kaggle/working/pipeline")
if target.is_symlink():
    target.unlink()
elif target.exists():
    shutil.rmtree(target)
os.symlink(found.resolve(), target)
print("Đã symlink -> /kaggle/working/pipeline ->", found.resolve())

Path("/kaggle/working/pipeline/work").mkdir(parents=True, exist_ok=True)

## Bước 4 — Tải dataset từ Google Drive

Điền link chia sẻ Google Drive (chế độ "Anyone with the link") vào `GDRIVE_URL` bên
dưới — file phải là **1 file .zip** chứa `Dataset/VAI_NVS_DATA_ROUND2/<scene>/...`
(zip nguyên thư mục `Dataset`, hoặc chỉ riêng `VAI_NVS_DATA_ROUND2` cũng được — cell
dưới tự dò tìm thư mục chứa các scene ở bất kỳ độ sâu nào trong zip, KHÔNG bắt buộc
đúng tên thư mục bọc ngoài, xem `docs/PORTED_KNOWLEDGE.md` mục 1).

In [ ]:
GDRIVE_URL = "https://drive.google.com/file/d/178EL7jCSVD59q19SMpeOgnOfOIC66I_t/view?usp=drive_link"

assert GDRIVE_URL, "Chưa điền GDRIVE_URL — dán link chia sẻ Google Drive (Anyone with the link) của file zip dataset vào biến này rồi chạy lại cell."

import os
os.makedirs("/kaggle/working/_dataset_raw", exist_ok=True)
!gdown --fuzzy "{GDRIVE_URL}" -O /kaggle/working/dataset.zip
!unzip -q -o /kaggle/working/dataset.zip -d /kaggle/working/_dataset_raw
print("Đã giải nén xong, đang dò tìm thư mục chứa các scene ...")

In [ ]:
# Tự dò thư mục chứa các scene phẳng (HCM0421/, chair/, bonsai/...) ở bất kỳ đâu
# trong zip vừa giải nén, rồi symlink về đúng vị trí mà pipeline/common/scenes.py
# cần: /kaggle/working/Dataset/VAI_NVS_DATA_ROUND2
#
# KHÔNG bắt buộc thư mục bọc ngoài phải tên đúng "VAI_NVS_DATA_ROUND2" — chỉ cần
# TÌM ĐƯỢC 1 thư mục (kể cả chính gốc giải nén, nếu zip không có lớp bọc ngoài)
# chứa đủ NHIỀU scene mong đợi trực tiếp bên trong. Logic này đã được sửa (fix)
# thành linh hoạt ở repo tiền nhiệm — bản cũ bắt buộc đúng tên thư mục nên sẽ báo
# lỗi "Không tìm thấy..." nếu file zip giải nén ra không có đúng lớp thư mục tên
# "VAI_NVS_DATA_ROUND2" đó (vd giải nén thẳng ra HCM0421/ ở gốc, hoặc thư mục bọc
# ngoài đặt tên khác) — dù dữ liệu vẫn đầy đủ. Đây là bản đã fix, port nguyên vẹn.
#
# Danh sách tên scene lặp lại thủ công ở đây (không import common.scenes) vì
# sys.path chưa trỏ tới pipeline/ ở bước này (việc đó làm ở cell kiểm tra ngay
# sau) — giữ đồng bộ với BTS_SCENES/GENERIC_SCENES trong pipeline/common/scenes.py
# nếu sau này thêm/bớt scene.
import os
from pathlib import Path

_expected_scene_dirs = {"HCM0421", "HCM0539", "HCM0540", "HCM0644", "HCM0674", "bonsai", "chair"}
_MIN_MATCH = 4  # đủ scene trùng khớp để tin đây đúng là thư mục dataset (tránh khớp nhầm thư mục rác)

RAW_ROOT = Path("/kaggle/working/_dataset_raw")
found = None
best_match = 0
for dirpath, dirnames, filenames in os.walk(RAW_ROOT):
    # __MACOSX/ là rác do Mac tạo khi nén zip — nó TỰ NHÂN BẢN y hệt cấu trúc thư
    # mục thật (HCM0421/, train/images/...) nhưng file bên trong chỉ là file
    # rác metadata "._<tên file>", không phải dữ liệu thật. Phải loại trừ, nếu
    # không os.walk có thể tìm trúng "__MACOSX/..." trước bản thật.
    dirnames[:] = [d for d in dirnames if d != "__MACOSX" and not d.startswith(".")]
    n_match = len(_expected_scene_dirs & set(dirnames))
    if n_match > best_match:
        best_match = n_match
        found = Path(dirpath)
    if n_match == len(_expected_scene_dirs):
        break  # khớp đủ cả 7 — dừng sớm, khỏi walk tiếp cho nhanh

if found is None or best_match < _MIN_MATCH:
    raise SystemExit(
        f"Không tìm thấy thư mục nào chứa >= {_MIN_MATCH}/{len(_expected_scene_dirs)} scene mong đợi "
        f"bên trong {RAW_ROOT}. Khớp tốt nhất: {found} ({best_match} scene). "
        f"Kiểm tra lại file zip GDRIVE_URL có đúng dataset không."
    )

print(f"Tìm thấy thư mục dataset tại: {found} ({best_match}/{len(_expected_scene_dirs)} scene khớp)")

target_parent = Path("/kaggle/working/Dataset")
target_parent.mkdir(parents=True, exist_ok=True)
target = target_parent / "VAI_NVS_DATA_ROUND2"
if target.is_symlink() or target.exists():
    if target.is_symlink():
        target.unlink()
    else:
        import shutil
        shutil.rmtree(target)
os.symlink(found.resolve(), target)
print("Đã symlink ->", target, "->", found.resolve())

os.environ["BTS_DATASET_ROOT"] = str(target)
print("BTS_DATASET_ROOT =", os.environ["BTS_DATASET_ROOT"])

In [ ]:
# Kiểm tra lại: liệt kê đủ 7 scene + scene nào có sparse hợp lệ — dataset đầy đủ
# thì kỳ vọng has_valid_provided_sparse=True cho CẢ 7 scene.
# Nếu train_ok=False hết cho mọi scene, kiểm tra lại BTS_DATASET_ROOT ở cell trên
# có trỏ đúng chỗ chứa VAI_NVS_DATA_ROUND2 hay không (thường do dataset.zip
# tải/giải nén thiếu — thử xoá /kaggle/working/_dataset_raw và tải lại từ đầu).
import sys
sys.path.insert(0, "/kaggle/working/pipeline")
from common.scenes import all_scenes, DATASET_ROOT

print("DATASET_ROOT =", DATASET_ROOT, "| tồn tại:", DATASET_ROOT.exists())
for s in all_scenes():
    ok_train = s.train_images_dir.exists()
    ok_csv = s.test_poses_csv.exists()
    n_train = len(list(s.train_images_dir.glob("*"))) if ok_train else 0
    print(f"{s.name:10s} {s.domain:8s} train_ok={ok_train} n_train={n_train:4d} "
          f"csv_ok={ok_csv} has_valid_provided_sparse={s.has_valid_provided_sparse()}")

## Bước 5 — Cấu hình + train 1 scene (baseline Vòng 1)

**Đổi các biến ở cell dưới rồi Save Version** (mỗi lần 1 tổ hợp scene+mode, ở version khác nhau):

- `SCENE`: 1 trong 7 tên — `HCM0421`, `HCM0539`, `HCM0540`, `HCM0644`, `HCM0674`, `bonsai`, `chair`.
- `MODE`:
  - `"holdout"` — train nhanh trên phần ảnh train KHÔNG bị giữ lại làm holdout, render
    + chấm điểm (Score) trên phần holdout đó. **Dùng để xem baseline hoạt động tốt tới
    đâu trước khi train bản nộp — checkpoint ra từ chế độ này TUYỆT ĐỐI KHÔNG được dùng
    để nộp bài** (chỉ train trên 1 phần dữ liệu).
  - `"final"` — train trên **100% ảnh train**, render `test_poses.csv` thật. Đây là
    checkpoint Vòng 1 dùng làm điểm khởi đầu cho Vòng 2+ (và có thể dùng để nộp bài
    luôn nếu không chạy Vòng 2+).
- `ANTIALIASING` (0/1, mặc định 1): mip-splatting antialiasing — đã đo có lợi thật ở
  repo tiền nhiệm (`docs/PORTED_KNOWLEDGE.md` mục 2), giữ mặc định BẬT.

Vòng 1 KHÔNG hỗ trợ depth-prior/exposure-comp/antenna-focus (xem đầu notebook) — nếu
cần thử các kỹ thuật đó, dùng notebook Vòng 2+ sau khi có checkpoint Vòng 1.

Chi tiết đầy đủ ghi ra file `pipeline/work/<scene>/02_train_baseline.log`. Xem tiến độ
lúc train đang chạy: mở 1 cell khác gõ
`!tail -n 30 /kaggle/working/pipeline/work/<scene>/02_train_baseline.log`.

In [ ]:
SCENE = "HCM0421"  # <-- đổi thành tên scene muốn train ở version này
MODE = "holdout"   # "holdout" (đo Score, KHÔNG dùng nộp bài) hoặc "final" (100% data, checkpoint Vòng 1 thật)

ANTIALIASING = 1   # 0/1 — mip-splatting antialiasing, mặc định BẬT (đã đo có lợi)

assert MODE in ("holdout", "final"), 'MODE phải là "holdout" hoặc "final"'

# In BANNER dễ thấy — repo tiền nhiệm từng có lần chạy thật quên đổi biến trước
# khi chạy cả notebook (~1 tiếng GPU Kaggle), ra nhầm dữ liệu dán nhầm tên. Đọc kỹ
# dòng dưới TRƯỚC KHI chạy tiếp các cell sau.
print("=" * 78)
print(f"  SCENE={SCENE}  MODE={MODE}  ANTIALIASING={ANTIALIASING}")
print("  --> KIỂM TRA LẠI ĐÚNG CẤU HÌNH ĐỊNH CHẠY TRƯỚC KHI CHẠY CÁC CELL TIẾP THEO! <--")
print("=" * 78)

In [ ]:
import os

if MODE == "holdout":
    # Tạo holdout nếu chưa có (script tự báo lỗi rõ ràng + gợi ý --overwrite nếu đã
    # tồn tại — không phải lỗi, version trước có thể đã tạo rồi, bỏ qua an toàn).
    holdout_dir = f"/kaggle/working/pipeline/work/{SCENE}/holdout"
    if not os.path.isdir(holdout_dir):
        !python /kaggle/working/pipeline/scripts/00_make_holdout_split.py --scene {SCENE}
    else:
        print(f"Đã có {holdout_dir} — bỏ qua tạo lại (dùng --overwrite thủ công nếu muốn tạo lại).")
    !python /kaggle/working/pipeline/scripts/01_run_colmap.py --scene {SCENE} --holdout
else:
    !python /kaggle/working/pipeline/scripts/01_run_colmap.py --scene {SCENE}

In [ ]:
import os

# holdout: iteration ít hơn (đủ tín hiệu đo Score, tiết kiệm quota GPU Kaggle nếu
# lặp lại nhiều scene) — final: 30000 (mặc định repo, chất lượng cao nhất cho
# checkpoint Vòng 1 thật, chỉ chạy 1 lần/scene).
os.environ["ITERATIONS"] = "15000" if MODE == "holdout" else "30000"
os.environ["ANTIALIASING"] = str(ANTIALIASING)

print("ITERATIONS =", os.environ["ITERATIONS"])

!bash /kaggle/working/pipeline/scripts/02_train_baseline.sh {SCENE}

In [ ]:
if MODE == "holdout":
    poses_csv = f"/kaggle/working/pipeline/work/{SCENE}/holdout/holdout_poses.csv"
    out_dir = f"/kaggle/working/pipeline/work/{SCENE}/holdout_renders"
    !python /kaggle/working/pipeline/scripts/03_render_test_poses.py --scene {SCENE} \
        --poses_csv {poses_csv} --out_dir {out_dir}
else:
    !python /kaggle/working/pipeline/scripts/03_render_test_poses.py --scene {SCENE}

In [ ]:
if MODE == "holdout":
    !python /kaggle/working/pipeline/scripts/04_eval_metrics.py --scene {SCENE}
else:
    print("MODE=final — không có ảnh GT để chấm điểm (dùng 100% ảnh train). "
          "Điểm tham khảo cho scene này lấy từ lần chạy MODE=holdout trước đó.")

### Xem thử vài ảnh render ra (kiểm tra bằng mắt)

Kiểm tra hợp lý (không nhiễu loạn, không sai màu/hình dạng bất thường) trước khi lưu
lên Drive. Nếu `MODE="holdout"`, Score định lượng đã in ở cell trên rồi — đây chỉ là
kiểm tra bằng mắt bổ sung.

In [ ]:
from pathlib import Path
from IPython.display import display
from PIL import Image

renders_dir = Path(f"/kaggle/working/pipeline/work/{SCENE}/holdout_renders" if MODE == "holdout"
                    else f"/kaggle/working/pipeline/work/{SCENE}/renders")
all_renders = sorted(renders_dir.glob("*.png"))
sample = all_renders[:4]
print(f"{len(all_renders)} ảnh render tại {renders_dir}, xem thử {len(sample)} ảnh đầu:")
for p in sample:
    display(Image.open(p))

## Bước 6 — Lấy checkpoint Vòng 1 để lưu lên Google Drive

**Chỉ làm bước này khi `MODE = "final"`.** Nếu vừa chạy `MODE = "holdout"`, checkpoint
vừa train chỉ dùng để đo Score, **KHÔNG tải lên Drive / KHÔNG dùng để nộp bài / KHÔNG
dùng làm input Vòng 2+** (chỉ train trên 1 phần ảnh train, thiếu dữ liệu so với bản
`final`).

Quy trình đầy đủ cho MỖI scene:
1. Chạy 1 (hoặc vài) version với `MODE="holdout"` để xem baseline có ra Score hợp lý
   không (đối chiếu `docs/PORTED_KNOWLEDGE.md` để biết khoảng Score đã từng đo được).
2. Chạy 1 version cuối với `MODE="final"` — đây là checkpoint Vòng 1 chính thức, làm
   input cho Vòng 2+ (`kaggle_round2_refine.ipynb`) hoặc dùng để nộp bài trực tiếp nếu
   không chạy thêm vòng nào.

**QUAN TRỌNG — PHẢI tải NGUYÊN thư mục `gs_model/`, KHÔNG chỉ file `.ply`** (xem
`docs/PORTED_KNOWLEDGE.md` mục 4): thư mục này còn chứa `cfg_args` (sh_degree lúc
train) và `pipeline_train_flags.json` (antialiasing thật đã dùng, xem
`docs/PORTED_KNOWLEDGE.md` mục 2) — thiếu 1 trong 2 file này, Vòng 2+ (hoặc bước
render/nộp bài) sẽ tự đoán SAI cấu hình lúc train mà KHÔNG báo lỗi rõ, làm méo hoàn
toàn PSNR/SSIM/LPIPS. Không cần nén gì — tải thẳng cả thư mục rồi upload nguyên vậy
lên Drive.

Checkpoint (trọng số đã train) nằm ở:
`pipeline/work/<SCENE>/gs_model/point_cloud/iteration_30000/point_cloud.ply`
(kèm 2 checkpoint giữa chừng ở `iteration_7000/` và `iteration_15000/`, phòng khi cần
iteration cuối bị lỗi).

Cách lấy: bấm **Save Version**, vào tab **Output**, tìm đúng thư mục
`pipeline/work/<SCENE>/gs_model/` (không cần lấy nguyên `/kaggle/working` — phần còn lại
chỉ là code/dataset/repo clone, không cần cho Vòng 2+/submission), tải thư mục này về
máy rồi upload thẳng lên Google Drive dưới dạng 1 thư mục — đặt tên thư mục trên Drive
rõ theo tên scene + vòng (vd `<SCENE>_round1_gs_model`) để không nhầm lẫn khi điền link
ở notebook Vòng 2+. Nhớ đổi chế độ share thư mục đó thành "Anyone with the link".

Lặp lại (holdout x N lần thử + final x 1 lần) cho cả 7 scene.